# Developing helper functions for the pipeline

In [1]:
import pandas as pd

df = pd.read_csv('../../data/raw/hyperbook.csv')

In [3]:
title, content = df[['title', 'content']]

In [3]:
from keybert import KeyBERT
from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import Counter
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from itertools import chain
import numpy as np

model_name = 'Qwen/Qwen3-Embedding-0.6B'

model = SentenceTransformer(model_name)
keybert = KeyBERT(model=model)
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(model_name='gpt-4', chunk_size=1024, chunk_overlap=128)

In [29]:
import pandas as pd

df = pd.read_csv('../../data/raw/hyperbook.csv')
sample = df.iloc[0]

In [ ]:
def parse_keywords(data):
    """
    Extracts the keywords from results
    generated by keybert in a recursive manner.
    """
    results = []

    def recurse(item):
        if isinstance(item, tuple):
            results.append(item[0])
        elif isinstance(item, list):
            for subitem in item:
                recurse(subitem)

    recurse(data)
    return results


def get_keywords(doc: str) -> list[str]:
        """
        Splits and embeds doc and returns the top 10
        most semantically relevant keywords
        """
        chunks = splitter.split_text(doc)
        result = keybert.extract_keywords(docs=chunks, top_n=10, nr_candidates=20)        

        keywords = parse_keywords(result)
        pooled = np.mean(model.encode(chunks), axis=0)

        kw_similarities = {}

        for kw, kw_emb in zip(keywords, model.encode(keywords)):
            distance = cos_sim(pooled, kw_emb)
            kw_similarities[kw] = float(distance)

        kw_similarities = dict(sorted(
            kw_similarities.items(), 
            key=lambda item: item[-1], 
            reverse=True
        ))

        top_keywords = list(kw_similarities.keys())[:10]
        
        return top_keywords

In [33]:
get_keywords(sample['content'])

[[('chlamydomonas', 0.5811), ('chlorophytes', 0.5608), ('chloroplasts', 0.5007), ('algae', 0.4876), ('eukaryotes', 0.4647), ('biflagellate', 0.4545), ('streptophytes', 0.4471), ('prokaryotic', 0.4449), ('organism', 0.4415), ('eukaryotic', 0.4306)], [('chlamydomonas', 0.5033), ('chloroplasts', 0.4518), ('photosensory', 0.4217), ('photoreceptor', 0.4216), ('organism', 0.4033), ('lenses', 0.3985), ('organelle', 0.3921), ('refractive', 0.3901), ('lens', 0.3839), ('eukaryotic', 0.3829)], [('chlamydomonas', 0.4818), ('algae', 0.4103), ('multicellular', 0.4085), ('photoreceptor', 0.3784), ('organism', 0.3683), ('photosynthetic', 0.3681), ('unicellular', 0.3545), ('photosynthesis', 0.3472), ('organisms', 0.3414), ('cells', 0.3407)], [('photoinhibition', 0.4663), ('photosynthetic', 0.4255), ('photosynthesis', 0.4096), ('chlorophyll', 0.4086), ('chlamydomonas', 0.3998), ('absorption', 0.3599), ('overexcite', 0.3413), ('oxidative', 0.3329), ('transduction', 0.3236), ('photosensor', 0.3176)], [('f

['chlamydomonas',
 'ciliates',
 'photopsins',
 'photopsin',
 'flagellar',
 'chlorophytes',
 'flagella',
 'chloroplasts',
 'cilium',
 'phototaxis']

In [2]:
import pandas as pd

sample = pd.read_csv('../../data/raw/hyperbook.csv').iloc[0]
content = sample['content']
chunks = splitter.split_text(content)

In [3]:
keywords = keybert.extract_keywords(chunks)
doc_embeddings, word_embeddings = keybert.extract_embeddings(chunks)

In [5]:
import numpy as np
pooled = np.mean(doc_embeddings, axis=0)

In [8]:
from sentence_transformers.util import cos_sim
cos_sim(pooled, word_embeddings[0])

tensor([[0.2572]])

In [13]:
pooled = np.mean(model.encode(chunks), axis=0)

In [16]:
flat = set([pair[0] for group in keywords for pair in group])

In [25]:
kw_similarities = {}

for kw, kw_emb in zip(flat, model.encode(list(flat))):
    distance = cos_sim(pooled, kw_emb)
    kw_similarities[kw] = float(distance)

In [31]:
sorted(kw_similarities.items(), key=lambda item: item[-1], reverse=True)[:10]

[('chlamydomonas', 0.6032787561416626),
 ('photopsins', 0.5817863345146179),
 ('photopsin', 0.5808806419372559),
 ('flagellar', 0.5654821395874023),
 ('chlorophytes', 0.5634056329727173),
 ('flagella', 0.559533953666687),
 ('chloroplasts', 0.557555079460144),
 ('cilium', 0.5517497062683105),
 ('phototaxis', 0.5438132882118225),
 ('chloroplast', 0.5377283096313477)]

In [ ]:
def get_keywords(doc: str) -> list[str]:
        """
        Splits and embeds doc and returns the top 10
        most semantically relevant keywords
        """
        chunks = splitter.split_text(doc)
        result = keybert.extract_keywords(docs=chunks)

        pooled = np.mean(model.encode(chunks), axis=0)
        flat = list(set([pair[0] for group in result for pair in group]))

        kw_similarities = {}

        for kw, kw_emb in zip(flat, model.encode(list(flat))):
                distance = cos_sim(pooled, kw_emb)
                kw_similarities[kw] = float(distance)

        

        return top_keywords

In [32]:
keybert.extract_keywords(content, top_n=10)

[('chlamydomonas', 0.627),
 ('chlorophytes', 0.613),
 ('ciliates', 0.6059),
 ('photopsin', 0.5924),
 ('photopsins', 0.5918),
 ('chloroplasts', 0.5802),
 ('chlorophyll', 0.5722),
 ('phototaxis', 0.5663),
 ('chloroplast', 0.5656),
 ('algae', 0.5645)]

In [38]:
keyword_similarities = {}

for kw, kw_emb in zip(keywords, word_embeddings):
    # Avoid unnecessary computations
    if kw not in keyword_similarities:
        cos_distance = cos_sim(pooled, kw_emb)
        keyword_similarities[kw] = kw_emb

In [40]:
keyword_similarities['chlamydomonas']

array([-0.01596837, -0.04105965, -0.01681677, ...,  0.02255882,
       -0.02031889, -0.00774533], dtype=float32)

In [ ]:
from sentence_transformers import SentenceTransformer
from collections import defaultdict
from hdbscan import HDBSCAN

model_name = 'Qwen/Qwen3-Embedding-0.6B'
model = SentenceTransformer(model_name)
clusterer = HDBSCAN(min_cluster_size=2, metric='euclidean')

def cluster_keywords(keywords: list[str]) -> dict:
    """
    Clusters keywords into semantic clusters
    and returns a map of their members
    """
    embeddings = model.encode(keywords)
    clusters = clusterer.fit_predict(embeddings)
    cluster_map = defaultdict(list)
    embeddings_map = {}

    for keyword, cluster_id in zip(keywords, clusters):
        if cluster_id != -1: # Skip noise if using HDBSCAN
            cluster_map[int(cluster_id)].append(keyword)
            embeddings_map[keyword] = embeddings[keywords.index(keyword)]

    return cluster_map, embeddings_map

In [14]:
model(['glycoproteins', 'photoproteins', 'opsins', 'proteins', 'sarcomeres'],return_tensors=True)

[tensor([[[ 1.6178e+00, -1.0052e+01, -6.7052e-02,  ..., -7.5931e+00,
           -1.1953e+01,  2.3351e-01],
          [-2.1009e+00,  4.4076e+00, -9.5821e-01,  ..., -7.1411e-01,
            2.8401e+00, -2.1420e-03],
          [-3.2621e+00,  6.0080e-01, -6.7357e-01,  ..., -1.3374e+00,
           -1.5131e-01,  1.3732e+00],
          [-3.1859e+00, -7.7886e+00, -1.1822e+00,  ..., -4.8352e-01,
            7.4725e-01, -2.3412e+00],
          [-4.0773e+00, -2.6724e+00, -1.3649e+00,  ..., -2.8752e-01,
           -1.4321e+00,  1.5707e+00]]]),
 tensor([[[  1.4083,  -7.3270,  -0.1671,  ...,  -6.3401, -11.9118,  -0.1040],
          [  1.5366,   6.6439,  -1.1439,  ...,  -1.9285,   2.8687,   0.6513],
          [  1.7761,   5.9108,  -0.4072,  ...,  -4.7630,  -2.8647,  -1.0821],
          [  1.3204,  -2.8606,  -0.5544,  ...,  -2.8091,   2.0338,  -2.9993],
          [ -2.1045,   0.8682,  -1.1636,  ...,   0.0126,  -1.5395,   0.3170]]]),
 tensor([[[  2.1722,  -6.6370,  -0.1050,  ...,  -7.5446, -13.5824,   

In [18]:
from transformers import pipeline

model_name = 'Qwen/Qwen3-Embedding-0.6B'
model = pipeline(task='feature-extraction', model=model_name, framework='pt')

def cluster_keywords(keywords: list[str]) -> dict:
    """
    Clusters keywords into semantic clusters
    and returns a map of their members
    """
    embeddings = model(keywords)
    clusters = clusterer.fit_predict(embeddings)
    cluster_map = defaultdict(list)
    embeddings_map = {}

    for keyword, cluster_id in zip(keywords, clusters):
        if cluster_id != -1: # Skip noise if using HDBSCAN
            cluster_map[int(cluster_id)].append(keyword)
            embeddings_map[keyword] = embeddings[keywords.index(keyword)]

    return cluster_map, embeddings_map

Device set to use cuda:0


In [22]:
keywords = ['glycoproteins', 'photoproteins', 'opsins', 'proteins', 'sarcomeres']
model(keywords, return_tensors=True)

[tensor([[[ 1.6178e+00, -1.0052e+01, -6.7052e-02,  ..., -7.5931e+00,
           -1.1953e+01,  2.3351e-01],
          [-2.1009e+00,  4.4076e+00, -9.5821e-01,  ..., -7.1411e-01,
            2.8401e+00, -2.1420e-03],
          [-3.2621e+00,  6.0080e-01, -6.7357e-01,  ..., -1.3374e+00,
           -1.5131e-01,  1.3732e+00],
          [-3.1859e+00, -7.7886e+00, -1.1822e+00,  ..., -4.8352e-01,
            7.4725e-01, -2.3412e+00],
          [-4.0773e+00, -2.6724e+00, -1.3649e+00,  ..., -2.8752e-01,
           -1.4321e+00,  1.5707e+00]]]),
 tensor([[[  1.4083,  -7.3270,  -0.1671,  ...,  -6.3401, -11.9118,  -0.1040],
          [  1.5366,   6.6439,  -1.1439,  ...,  -1.9285,   2.8687,   0.6513],
          [  1.7761,   5.9108,  -0.4072,  ...,  -4.7630,  -2.8647,  -1.0821],
          [  1.3204,  -2.8606,  -0.5544,  ...,  -2.8091,   2.0338,  -2.9993],
          [ -2.1045,   0.8682,  -1.1636,  ...,   0.0126,  -1.5395,   0.3170]]]),
 tensor([[[  2.1722,  -6.6370,  -0.1050,  ...,  -7.5446, -13.5824,   

In [12]:
model

In [14]:
keywords = ['glycoprotein', 'protein', 'glycoproteins', 'mussels', 'clams', 'mollusks', 'robot', 'android', 'artificial intelligence']

cluster_keywords(keywords)

d:\Projects\hyperbook-weaver\venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
d:\Projects\hyperbook-weaver\venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


(defaultdict(list,
             {0: ['glycoprotein', 'protein', 'glycoproteins'],
              1: ['mussels', 'clams', 'mollusks']}),
 {'glycoprotein': array([-0.03381737, -0.02381012, -0.01595601, ...,  0.00041461,
         -0.01179637,  0.01176754], dtype=float32),
  'protein': array([-0.00459065, -0.04338182, -0.01411273, ..., -0.01708603,
         -0.00218913,  0.03300722], dtype=float32),
  'glycoproteins': array([-0.04787508, -0.03137838, -0.01602631, ..., -0.00337595,
         -0.01681524,  0.01844322], dtype=float32),
  'mussels': array([-0.06903044, -0.0236024 , -0.01246766, ...,  0.00068482,
         -0.00723394, -0.01174365], dtype=float32),
  'clams': array([-0.05949921, -0.00083166, -0.01345477, ..., -0.01180673,
         -0.00942874, -0.01803245], dtype=float32),
  'mollusks': array([-0.0519052 ,  0.00618933, -0.01111559, ...,  0.00023069,
         -0.00674414,  0.02556807], dtype=float32)})

In [ ]:
import pandas as pd

df = pd.read_csv('../../data/raw/hyperbook.csv', index_col='url')
sample = df.iloc[0]
content = sample['content']

In [2]:
import plotly.graph_objects as go
import numpy as np

# Sample data
x = np.random.rand(50)
y = np.random.rand(50)
z = np.random.rand(50)

# Create a 3D scatter trace
trace = go.Scatter3d(
    x=x, y=y, z=z,
    mode='markers',
    marker=dict(size=5, color=z, colorscale='Viridis', opacity=0.8),
)

# Create a figure and add the trace
fig = go.Figure(data=[trace])

# Customize layout
fig.update_layout(
    title='Simple 3D Scatter Plot',
    scene=dict(
        xaxis_title='X-axis',
        yaxis_title='Y-axis',
        zaxis_title='Z-axis'
    ),
    height=1000
)

# Display the plot
fig.show()

In [18]:
from sentence_transformers import SentenceTransformer, util

model_name = 'Qwen/Qwen3-Embedding-0.6B'

def name_cluster(keywords: list[str]) -> str:
    """
    Names a cluster of keywords by finding the keyword
    closest to the centroid of the embeddings.
    """

    model = SentenceTransformer(model_name)
    embeddings = model.encode(keywords, convert_to_tensor=True, normalize_embeddings=True)

    # Calculate the arithmetic mean and mormalize
    centroid = util.normalize_embeddings(embeddings.mean(dim=0, keepdim=True))
    # Get the embedding closest to the centroid
    name_index = int(util.cos_sim(centroid, embeddings)[0].argmax())

    return keywords[name_index]

In [19]:
keywords = ['glycoprotein', 'protein', 'glycoproteins']

In [20]:
name_cluster(keywords)

'glycoproteins'

In [25]:
import pandas as pd

df = pd.read_csv('../../data/raw/hyperbook.csv')
titles = df['title']
extracted_keywords = df['extracted_keywords'].apply(lambda s: s.split(','))

In [28]:
import networkx as nx

# Map each keyword to its cluster
keyword_to_cluster = {
    keyword: cluster_id
    for cluster_id, keywords in cluster_map.items()
    for keyword in keywords
}

G = nx.Graph()

# Add cluster nodes
for cluster_id, cluster_name in cluster_names.items():
    G.add_node(f'cluster_{cluster_id}',  label=cluster_name + ' (topic)', type='cluster')

# Add article nodes
for i, pair in enumerate(zip(titles, extracted_keywords)):
    title, keywords = pair[0], pair[1]
    G.add_node(f'article_{i}', label=title + ' (article)', type='article')

    added_clusters = set()
    
    for kw in keywords:
        if kw in keyword_to_cluster:
            cluster_id = keyword_to_cluster[kw]
            # Add edge between cluster and article
            if cluster_id and cluster_id not in added_clusters:
                G.add_edge(f'article_{i}', f'cluster_{cluster_id}')    

In [33]:
from pyvis import network as net
import networkx as nx

g = net.Network(notebook=True)
g.force_atlas_2based()
g.from_nx(G)
g.show('example.html')

example.html
